# 🧠【RAG01】打造向量資料庫 —— AI 晶片與小晶片 (Chiplet) 前沿架構研發知識庫

> **課程**：生成式 AI 實作課程　|　**第七週作業 1020**
> **學號**：114521045　|　**系級**：電子組碩二　|　**姓名**：楊荏鈞
> **主題**：打造自己的 RAG 對話機器人（程式一：讀取並向量化文本資料，建立向量資料庫）

---
### 📘 本週專案背景與知識來源說明
本專案聚焦於下世代高效能 AI 加速晶片架構前沿研究，選取硬體研發核心文獻與架構設計手冊作為知識來源：
1. **`01_DCIM_partition_methods.txt`**：SRAM 數位存算一體 (Digital Compute-In-Memory) 分割方法查表手冊（涵蓋 Bitcell、Subarray、Macro、Tile、Chiplet 多層級分割）。
2. **`02_DCIM_vs_TEMP_partition_verdict.txt`**：DCIM 與晶圓級晶片 (Wafer-Scale Chips, WSC) TEMP 張量分割機制之移植性裁決報告。
3. **`03_wang-tc25-slides-draft.txt`**：IEEE TC 2025 頂尖文獻《On Task Mapping in Multi-chiplet Based Many-Core Systems》導讀草稿（兩階段 BQP 任務映射與 Inter/Intra-Chiplet 通訊優化）。
4. **`04_IEEE_AI晶片與小晶片前沿研究文獻總覽與技術問答指南.txt`**：統合 NoC 無死鎖路由、TriCIM 容量感知、Simba MCM 等核心技術之全景導讀指南。
5. **`05_Chiplet_CNN_Task_Allocation.pdf`**：IEEE 論文原文《A Task Allocation Method for Chiplet-Based CNN Accelerators》。

### 1. 建立文檔資料夾並準備知識檔案
在此步驟中建立 `uploaded_docs` 資料夾，放置所有欲納入向量資料庫的文字與論文檔案。

In [ ]:
import os
upload_dir = "uploaded_docs"
os.makedirs(upload_dir, exist_ok=True)
print(f"✅ 資料夾已就緒，請將知識文本檔案放至： {upload_dir}")

### 2. 安裝必要套件並引入
安裝 LangChain、向量資料庫 FAISS、文本切分器、文件載入器（含 PyPDF 與 Word），以及 HuggingFace Sentence-Transformers 套件。

In [ ]:
# 安裝必要套件（完全符合老師固定指定規格）
!pip install -U langchain langchain-community pypdf python-docx faiss-cpu
!pip install -U sentence-transformers transformers huggingface_hub

In [ ]:
import os
from langchain_community.document_loaders import TextLoader, PyPDFLoader, UnstructuredWordDocumentLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

### 3. 依 Google 建議加入 EmbeddingGemma 前綴類別
採用 Google 官方推薦的 EmbeddingGemma 模型 (`google/embeddinggemma-300m`)，並遵照官方檢索慣例加入前綴：
- 文件向量格式：`title: {title|none} | text: ...`
- 查詢 Query 格式：`task: search result | query: ...`

In [ ]:
class EmbeddingGemmaEmbeddings(HuggingFaceEmbeddings):
    def __init__(self, **kwargs):
        super().__init__(
            model_name="google/embeddinggemma-300m",   # HF 官方開放模型
            encode_kwargs={"normalize_embeddings": True},  # 向量標準化檢索慣例
            **kwargs
        )

    def embed_documents(self, texts):
        # 文件向量：加入官方前綴以提升檢索表現
        texts = [f'title: none | text: {t}' for t in texts]
        return super().embed_documents(texts)

    def embed_query(self, text):
        # 查詢向量：官方推薦之 Retrieval-Query 前綴
        return super().embed_query(f'task: search result | query: {text}')

### 4. 載入 `uploaded_docs` 資料夾內的所有文獻
自動辨識並載入 `.txt`、`.pdf`、`.docx` 等格式文件，完整讀取晶片架構知識內容。

In [ ]:
folder_path = upload_dir
documents = []

for file in sorted(os.listdir(folder_path)):
    path = os.path.join(folder_path, file)
    if file.endswith(".txt") or file.endswith(".md"):
        loader = TextLoader(path, encoding="utf-8")
    elif file.endswith(".pdf"):
        loader = PyPDFLoader(path)
    elif file.endswith(".docx"):
        loader = UnstructuredWordDocumentLoader(path)
    else:
        continue
    
    loaded = loader.load()
    print(f"📄 已成功載入：{file}（頁數/段落數：{len(loaded)}）")
    documents.extend(loaded)

print(f"\n🎉 總共成功載入 {len(documents)} 份文檔！")

### 5. 文本切塊 (Chunking) 與 Hugging Face 認證
使用 `RecursiveCharacterTextSplitter` 將長文切分為小區塊 (`chunk_size=500`, `chunk_overlap=100`)，以保留完整語意上下文。

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
split_docs = splitter.split_documents(documents)
print(f"✂️ 文本切割完成！共產生 {len(split_docs)} 個知識片段 (Chunks)。")

# 若 Colab Secrets 設有 HuggingFace Token 則自動登入，否則以開放模式載入
try:
    from google.colab import userdata
    hf_token = userdata.get('HuggingFace')
    if hf_token:
        from huggingface_hub import login
        login(token=hf_token)
        print("🔑 Hugging Face Token 驗證成功！")
except Exception as e:
    print("ℹ️ 於非 Colab 環境或未設定 HF Token，採用直接載入模式。")

### 6. 建立 FAISS 向量資料庫
將切割後的 Chunks 透過 `EmbeddingGemmaEmbeddings` 轉換為高維度語意向量，並構建 FAISS 索引。

In [ ]:
print("🚀 正在向量化文本並建立 FAISS 向量資料庫...")
embedding_model = EmbeddingGemmaEmbeddings()
vectorstore = FAISS.from_documents(split_docs, embedding_model)
print("✨ 向量資料庫建立完成！")

### 7. 本地快速檢索驗證
以專業題目進行 Top-3 相關文檔檢索測試，驗證知識庫索引準確度。

In [ ]:
test_query = "在 DCIM 設計中，Macro 內部 array 切法與跨 tile/die 切法有何不同？"
docs = vectorstore.similarity_search(test_query, k=3)

print(f"🔍 測試檢索問題：{test_query}\n")
for i, doc in enumerate(docs, 1):
    print(f"--- 檢索結果 [{i}] (來源：{doc.metadata.get('source', '未知')}) ---")
    print(doc.page_content.strip()[:300] + "...\n")

### 8. 儲存向量資料庫並壓縮為 `rag-data.zip`（與 `faiss_db.zip`）
將索引存為本地資料夾 `faiss_db`，並封裝壓縮為助教與系統規定的 `rag-data.zip`（同時建立 `faiss_db.zip` 備份），以供第二份程式使用。

In [ ]:
# 儲存本地向量資料庫
vectorstore.save_local("faiss_db")

# 依作業規定壓縮為 rag-data.zip 與 faiss_db.zip
!zip -r faiss_db.zip faiss_db
!cp faiss_db.zip rag-data.zip

print("\n=======================================================")
print("✅ 向量資料庫已成功儲存為 'rag-data.zip' 與 'faiss_db.zip'！")
print("請將此壓縮檔上傳至 Google Drive 或 GitHub Release，")
print("並取得直連下載網址填入【RAG02】程式中，助教即可直接執行評分。")
print("=======================================================")